In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("johnsmith88/heart-disease-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'heart-disease-dataset' dataset.
Path to dataset files: /kaggle/input/heart-disease-dataset


In [2]:
import pandas as pd
import glob
import os

# 1. Locate the downloaded CSV file automatically
csv_files = glob.glob(os.path.join(path, "**/*.csv"), recursive=True)

if csv_files:
    # Load the first found CSV file into a DataFrame
    df = pd.read_csv(csv_files[0])
    print(f"Successfully loaded dataset from: {csv_files[0]}")
    print(f"Dataset Shape: {df.shape}\n")

    # 2. Check for missing values (Requirement 1)
    print("--- Missing Values Per Column ---")
    missing_data = df.isnull().sum()
    missing_percent = (df.isnull().sum() / len(df)) * 100
    missing_df = pd.DataFrame({'Total Missing': missing_data, 'Percentage (%)': missing_percent})
    print(missing_df[missing_df['Total Missing'] > 0] if missing_data.sum() > 0 else "No missing values found initially!")

    # 3. Display data types and basic summary
    print("\n--- DataFrame Info ---")
    df.info()
else:
    print("No CSV files found in the downloaded path. Please check the directory.")

Successfully loaded dataset from: /kaggle/input/heart-disease-dataset/heart.csv
Dataset Shape: (1025, 14)

--- Missing Values Per Column ---
No missing values found initially!

--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 5   fbs       1025 non-null   int64  
 6   restecg   1025 non-null   int64  
 7   thalach   1025 non-null   int64  
 8   exang     1025 non-null   int64  
 9   oldpeak   1025 non-null   float64
 10  slope     1025 non-null   int64  
 11  ca        1025 non-null   int64  
 12  thal      1025 non-null   int64  
 13  target    1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB


In [3]:
import numpy as np
import pandas as pd

# Copy data to preserve original inputs
processed_df = df.copy()

print("=========================================")
print("MODULE 1: INPUT - Neutralizing Outliers")
print("=========================================\n")

# Continuous numeric columns susceptible to extreme outliers
numerical_cols = ['trestbps', 'chol', 'thalach', 'oldpeak']

for col in numerical_cols:
    # 1. Calculate IQR bounds (As required by DecodeLabs)
    Q1 = processed_df[col].quantile(0.25)
    Q3 = processed_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Identify anomalies
    outliers_mask = (processed_df[col] < lower_bound) | (processed_df[col] > upper_bound)
    num_outliers = outliers_mask.sum()

    # 2. Impute outliers using the Median to preserve variance without skewing regression
    if num_outliers > 0:
        median_val = processed_df[col].median()
        processed_df.loc[outliers_mask, col] = median_val
        print(f"• Column '{col}': Imputed {num_outliers} outliers with median ({median_val})")
    else:
        print(f"• Column '{col}': No outliers detected.")

print("\n=========================================")
print("MODULE 2: PROCESS - Engineering Features")
print("=========================================\n")

# Requirement: Engineer at least 3 new predictive features from existing data columns
# Feature 1: Age Groups (Binned categorical profile mapped to numerical structural logic)
processed_df['age_group'] = pd.cut(
    processed_df['age'],
    bins=[0, 45, 60, 100],
    labels=[0, 1, 2] # 0: Young, 1: Middle-aged, 2: Senior
).astype(int)

# Feature 2: Cholesterol-to-Age Ratio (Captures age-adjusted metric progression)
processed_df['chol_age_ratio'] = processed_df['chol'] / (processed_df['age'] + 1)

# Feature 3: Heart Rate Reserve Estimate Proxy (Max Heart Rate 'thalach' minus Resting Blood Pressure 'trestbps')
processed_df['hr_bps_interaction'] = processed_df['thalach'] - processed_df['trestbps']

print("✓ Created Feature 1: 'age_group' (Binned lifecycle representation)")
print("✓ Created Feature 2: 'chol_age_ratio' (Continuous age-adjusted metabolic indicator)")
print("✓ Created Feature 3: 'hr_bps_interaction' (Cardiovascular performance metric variance)")
print(f"\nNew Pipeline Dataset Shape: {processed_df.shape}")

MODULE 1: INPUT - Neutralizing Outliers

• Column 'trestbps': Imputed 30 outliers with median (130.0)
• Column 'chol': Imputed 16 outliers with median (240.0)
• Column 'thalach': Imputed 4 outliers with median (152.0)
• Column 'oldpeak': Imputed 7 outliers with median (0.8)

MODULE 2: PROCESS - Engineering Features

✓ Created Feature 1: 'age_group' (Binned lifecycle representation)
✓ Created Feature 2: 'chol_age_ratio' (Continuous age-adjusted metabolic indicator)
✓ Created Feature 3: 'hr_bps_interaction' (Cardiovascular performance metric variance)

New Pipeline Dataset Shape: (1025, 17)


In [5]:
# Use the updated pandas-specific import syntax recommended by the warning
import pandera.pandas as pa
from pandera import Column, Check

print("=========================================")
print("MODULE 3: OUTPUT - Schema Validation Contract")
print("=========================================\n")

# Explicitly cast our numeric-turned-float columns to avoid strict type mismatch
float_cols = ['trestbps', 'chol', 'thalach', 'oldpeak', 'chol_age_ratio', 'hr_bps_interaction']
for col in float_cols:
    processed_df[col] = processed_df[col].astype(float)

# Define strict architectural boundaries for the processed dataset
heart_disease_schema = pa.DataFrameSchema(
    columns={
        "age": Column(int, Check.between(0, 120)),
        "sex": Column(int, Check.isin([0, 1])),
        "cp": Column(int, Check.between(0, 3)),
        "trestbps": Column(float, Check.greater_than_or_equal_to(0)),
        "chol": Column(float, Check.greater_than_or_equal_to(0)),
        "fbs": Column(int, Check.isin([0, 1])),
        "restecg": Column(int, Check.between(0, 2)),
        "thalach": Column(float, Check.greater_than_or_equal_to(0)),
        "exang": Column(int, Check.isin([0, 1])),
        "oldpeak": Column(float, Check.greater_than_or_equal_to(0)),
        "slope": Column(int, Check.between(0, 2)),
        "ca": Column(int, Check.between(0, 4)),
        "thal": Column(int, Check.between(0, 3)),
        "target": Column(int, Check.isin([0, 1])),

        # Validating our newly engineered features
        "age_group": Column(int, Check.isin([0, 1, 2])),
        "chol_age_ratio": Column(float, Check.greater_than_or_equal_to(0)),
        "hr_bps_interaction": Column(float)
    },
    strict=True,  # Enforces that ONLY these structural columns exist
    coerce=True   # Automatically ensures types line up safely during verification
)

try:
    # Perform coordinate optimization validation pass
    validated_df = heart_disease_schema.validate(processed_df)
    print("✓ SUCCESS: Production-ready dataset passed all Pandera schema contracts!")

    # Save the finalized clean high-fidelity fuel to a CSV file for submission
    validated_df.to_csv("heart_disease_cleaned_project1.csv", index=False)
    print("✓ ARTIFACT GENERATED: 'heart_disease_cleaned_project1.csv' saved successfully.")

except pa.errors.SchemaErrors as err:
    print("✗ Validation Error encountered:\n", err)

MODULE 3: OUTPUT - Schema Validation Contract

✓ SUCCESS: Production-ready dataset passed all Pandera schema contracts!
✓ ARTIFACT GENERATED: 'heart_disease_cleaned_project1.csv' saved successfully.


# Project 1: Advanced EDA & Feature Engineering
## Technical Architecture & Design Summary (IPO Blueprint)

This notebook implements an enterprise-grade Input-Process-Output (IPO) architecture to transform raw data into a mathematically clean, validated dataset optimized for machine learning algorithms.

---

### 1. Module 1: Input (Outlier Mitigation via Non-Parametric Boundaries)
* **Objective:** Identify and neutralize statistical anomalies within continuous numerical features (`trestbps`, `chol`, `thalach`, `oldpeak`).
* **Methodology:** Rather than executing row deletions—which reduces sample volume and introduces selection bias—this pipeline applies robust, non-parametric boundaries using the **Interquartile Range (IQR)**:
  $$\text{Lower Bound} = Q1 - 1.5 \times \text{IQR}$$
  $$\text{Upper Bound} = Q3 + 1.5 \times \text{IQR}$$
* **Outcome:** Successfully isolated and neutralized **57 extreme anomalies**, replacing them with global median values to stabilize variance while maintaining a robust sample size of 1,025 observations.

### 2. Module 2: Process (Vectorized Feature Engineering)
To enrich the feature space and provide high-fidelity predictive power for future modeling, three domain-specific features were engineered using vectorized mathematical operations:
* **`age_group` (Categorical Lifecycle Profile):** Bins continuous ages into structural logical tiers (`0`: Young, `1`: Middle-aged, `2`: Senior).
* **`chol_age_ratio` (Continuous Metabolic Indicator):** Captures age-adjusted cholesterol density progression.
* **`hr_bps_interaction` (Cardiovascular Performance Metric):** Measures the physiological spread between peak physical output (`thalach`) and resting cardiovascular pressure (`trestbps`).

### 3. Module 3: Output (Pandera Schema Validation Contract)
* **Objective:** Secure structural fidelity and enforce a strict data contract before exporting production artifacts.
* **Methodology:** Implemented the **Pandera** validation framework to programmatically audit data types, boundary constraints (e.g., categorical inclusions, non-negative limits), and dimensionality.
* **Outcome:** The pipeline successfully verified all 17 features against the data contract schema and exported the pristine asset as `heart_disease_cleaned_project1.csv`.